<a href="https://colab.research.google.com/github/riAnriAn0/EdD_1/blob/main/treinamento_yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CONSTANTES DO CÓDIGO

In [ ]:
!pip install roboflow ultralytics

In [ ]:
WORKSPACE = "rian-cy3ek"
PROJETO = "deteccao-pista-obb"
VERSAO_DATASET =4
API_KEY = "jMzXhDriyWMI8QA9LQwk"

In [ ]:

from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJETO)
version = project.version(VERSAO_DATASET)
dataset = version.download("yolov8-obb")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to deteccao-pista-obb-4 in yolov8-obb:: 100%|██████████| 122/122 [00:00<00:00, 11246.27it/s]


O código acima faz o download do dataset do roboflow.

**Para alterar o dataset deve ser alterado a versão do dataset ou nome do projeto**

In [ ]:
from ultralytics import YOLO

DATA_SET_PATH = f"{PROJETO}-{VERSAO_DATASET}/data.yaml"
EPOCHS = 100
IMAGE_SIZE = 160
BATCH_SIZE = 16

# ----------- Modelo para retreinamento -----------

MODEL_PATH = "runs/detect/train/weights/best.pt"

model = YOLO("best.pt")

results = model.train(
    data=DATA_SET_PATH,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
)

if results is not None: model = YOLO(f"{results.save_dir}/weights/best.pt")

exported_model = model.export(format="tflite", int8=True)



Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=deteccao-pista-obb-4/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=160, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=

In [ ]:
import shutil
import os
from google.colab import drive

NUM_TREINO = ""
CAMINHO_BASE = f"/content/runs{NUM_TREINO}/obb/train/weights"

# 1. Garante que o Drive está montado
drive.mount('/content/drive')

# 2. Define a pasta de destino no seu Drive (ajuste o nome se quiser)
caminho_destino = '/content/drive/MyDrive/Colab_Treinamento_yolo/modelos'

# Cria a pasta se ela não existir
if not os.path.exists(caminho_destino):
    os.makedirs(caminho_destino)

# 3. Lista dos arquivos que você quer salvar
arquivos_para_salvar = [f"{CAMINHO_BASE}/best.pt", f"{CAMINHO_BASE}/best_saved_model/best_full_integer_quant.tflite", f"{CAMINHO_BASE}/best_saved_model/best_int8.tflite"]

for arquivo in arquivos_para_salvar:
    # Verifica se o arquivo realmente foi gerado antes de tentar copiar
    if os.path.exists(arquivo):
        shutil.copy(arquivo, caminho_destino)
        print(f"Sucesso: {arquivo} copiado para o Drive!")
    else:
        print(f"Erro: O arquivo {arquivo} não foi encontrado no diretório local.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sucesso: /content/runs/obb/train/weights/best.pt copiado para o Drive!
Sucesso: /content/runs/obb/train/weights/best_saved_model/best_full_integer_quant.tflite copiado para o Drive!
Sucesso: /content/runs/obb/train/weights/best_saved_model/best_int8.tflite copiado para o Drive!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

pasta_destino = '/content/drive/MyDrive/Colab_Treinamento_yolo'

if not os.path.exists(pasta_destino):
    os.makedirs(pasta_destino)
    print(f"Pasta {pasta_destino} criada com sucesso!")

Pasta /content/drive/MyDrive/Colab_Treinamento_yolo criada com sucesso!


In [ ]:
from ultralytics import YOLO

model = YOLO("best.pt")
exported_model = model.export(format="tflite", int8=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'best.pt' with input shape (1, 3, 160, 160) BCHW and output shape(s) (1, 5, 525) (5.9 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx>=1.12.0,<2.0.0', 'onnx2tf>=1.26.3,<1.29.0', '